# H5 · Indefinite optimism and companies without plans (*Zero to One*, ch. 2, 6, 13)

Thiel's charge is that since the 1980s America has been *indefinitely optimistic*: investors expect the future to be better without knowing how, so money flows to process and options rather than to definite plans (ch. 6). After the dot-com crash, Silicon Valley's lesson was to stay lean and avoid plans; Thiel argues that a bad plan is better than no plan (ch. 2). Cleantech was his case study: optimistic investors funded firms with no specific plan, and the result was a bubble (ch. 13).

| Claim | Test | Data |
|---|---|---|
| **H5a.** Public markets fund companies without a working business model, and more so in optimistic periods. | Share of US IPOs with losses before listing, 1980–2025, and the SPAC wave. Did the years after the book look like 1999–2000? | Ritter's IPO statistics (Tables 9 and 15b) |
| **H5b.** Firms without a plan to profit do worse for their investors. | (i) Ritter: three-year returns of loss-making against profitable IPOs, 1980–2024. (ii) Our own check on new SEC filers listed 2013–2022: survival, value multiple and profitability three years later, by whether they made a profit on arrival. | Ritter Table 16b; SEC XBRL (the H1–H3 cache) |
| **H5c.** Cleantech fails investors even when the technology succeeds. | World solar generation since 2014 against the returns of clean-energy ETFs and solar manufacturers, from the book's date and from the 2007 cleantech peak. | Our World in Data energy dataset (GitHub), Yahoo Finance |

**Caveats**
* Ritter's figures are transcribed from his PDF into `reference/` (each file names its source table and date). They exclude penny stocks, unit offers, ADRs, REITs, banks and SPACs.
* "New SEC filers" are not the same as IPOs: they include uplistings, spin-offs and companies that merged with a SPAC. SPACs themselves (SIC 6770, or "Acquisition" in the name) are excluded.
* Losses before listing are not the same as having no plan: biotech and high-growth software firms lose money by design. That is why the size split in Ritter's table matters.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys; from pathlib import Path  # thiel_common.py lives in the repo root
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "thiel_common.py").exists())))
import thiel_common as tc
tc.style()
REF = tc.ROOT / "reference"
read = lambda f: pd.read_csv(REF / f, comment="#")
t9, t15, t16 = read("ritter_t9_unprofitable_ipos.csv"), read("ritter_t15b_spac_ipos.csv"), read("ritter_t16b_returns_by_profitability.csv")
print("synthetic mode" if tc.SYNTHETIC else "live data")
print(f"Ritter Table 9: {t9.year.min()}-{t9.year.max()}, {t9.n_ipos.sum():,} IPOs")

## H5a · How many IPOs had no profits?

In [ ]:
eras = {"1980-1994": (1980, 1994), "1995-2000 (dot-com)": (1995, 2000), "2001-2013": (2001, 2013),
        "2014-2021 (after the book)": (2014, 2021), "2022-2025": (2022, 2025)}
def wavg(d): return (d.pct_eps_negative * d.n_ipos).sum() / d.n_ipos.sum()
era = pd.DataFrame([{"era": k, "n_ipos": int(t9[(t9.year >= a) & (t9.year <= b)].n_ipos.sum()),
                     "pct_unprofitable": wavg(t9[(t9.year >= a) & (t9.year <= b)]),
                     "spacs": int(t15[(t15.year >= a) & (t15.year <= b)].n_spacs.sum()),
                     "spac_proceeds_bn": t15[(t15.year >= a) & (t15.year <= b)].proceeds_bn.sum()}
                    for k, (a, b) in eras.items()])
tc.save_table(era, "h5a_eras", index=False)
print("Years with 75% or more loss-making IPOs:", t9[t9.pct_eps_negative >= 75].year.tolist())
era.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
ax.axvspan(1998.5, 2000.5, color=tc.GREY, alpha=0.15); ax.text(1999.5, 96, "dot-com", ha="center", fontsize=8, color=tc.MUTED)
ax.axvline(2014.7, color=tc.GREY, lw=0.9); ax.text(2014.8, 96, " book", fontsize=8, color=tc.MUTED)
ax.plot(t9.year, t9.pct_eps_negative, color=tc.C[0], marker="o", ms=4, lw=1.8)
for y in (2000, 2014, 2018, 2021, 2025):
    v = t9.set_index("year").pct_eps_negative[y]; ax.annotate(f"{v:.0f}%", (y, v), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8,
                                                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
ax.set_ylim(0, 100); ax.set_ylabel("% of IPOs with negative earnings")
ax.set_title("US IPOs losing money before listing, 1980–2025", loc="left")
ax = axes[1]
yrs = t15.year
ax.bar(yrs - 0.2, t9.set_index("year").n_ipos.reindex(yrs).values, width=0.4, color=tc.C[0], label="Operating-company IPOs")
ax.bar(yrs + 0.2, t15.n_spacs, width=0.4, color=tc.C[1], label="SPAC IPOs")
ax.set_ylabel("Number of IPOs"); ax.legend(loc="upper left")
ax.set_title("The SPAC wave, 2003–2025", loc="left")
fig.text(0.01, -0.02, "Source: Jay R. Ritter, IPO Statistics, Tables 9 and 15b.", fontsize=8, color=tc.MUTED)
fig.tight_layout(); tc.save_fig(fig, "h5_fig1_unprofitable_ipos")

## H5b (i) · Do loss-making IPOs do worse? Ritter, 1980–2024

* **Market-adjusted:** the IPO's three-year buy-and-hold return minus the total stock market's over the same period.
* **Style-adjusted:** the return minus that of seasoned firms of similar size and book-to-market, which is the fairer comparison.

In [ ]:
print(t16.to_string(index=False))
fig, ax = plt.subplots(figsize=(8.5, 3.8))
grp = t16[t16.profitability != "All"].copy()
grp["label"] = grp.sales_group.replace({"All": "All IPOs"})
cats = ["All IPOs", "Sales < $100m", "Sales > $100m"]
x = np.arange(len(cats))
for i, prof in enumerate(["Unprofitable", "Profitable"]):
    v = grp[grp.profitability == prof].set_index("label").bhr3_style_adj_pct.reindex(cats)
    ax.bar(x + (i - 0.5) * 0.38, v, width=0.36, color=tc.C[1] if prof == "Unprofitable" else tc.C[0], label=prof)
    for xi, vi in zip(x, v): ax.text(xi + (i - 0.5) * 0.38, vi + (1 if vi >= 0 else -1), f"{vi:+.1f}%", ha="center", va="bottom" if vi >= 0 else "top", fontsize=8)
ax.axhline(0, color=tc.INK, lw=0.7); ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel("3-year return vs matched firms, %"); ax.legend(loc="lower right")
ax.set_title("Style-adjusted 3-year returns after IPO, 1980–2024 (Ritter Table 16b)", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h5_fig2_ritter_returns")

## H5b (ii) · Our own check: new SEC filers, 2013–2022

A firm enters the cohort in the first year it reports public float, provided it was absent from all earlier years from 2011 on. Profitability on arrival is net income in that year (or the next, if the first is missing). Outcomes are measured three years later.

The test uses data already cached by H1–H3. No new SEC downloads are needed except the annual NetIncomeLoss frames, which H2 already fetched.

In [ ]:
pf = tc.public_float_panel(range(2011, 2026))
first = pf.sort_values("year").groupby("cik").first()
sic = tc.sec_sic()
first["sic"] = first.index.map(sic)
spac = first.sic.eq("6770") | first.entityName.fillna("").str.contains(r"\bACQUISITION\b|\bSPAC\b|BLANK CHECK", case=False)
coh = first[(first.year >= 2013) & (first.year <= 2022) & ~spac].copy()
print(f"{len(first[(first.year >= 2013) & (first.year <= 2022)]):,} new filers 2013-2022; {int(spac[(first.year >= 2013) & (first.year <= 2022)].sum()):,} SPACs removed; {len(coh):,} kept")

ni = {y: tc.sec_frame("NetIncomeLoss", f"CY{y}").groupby("cik").val.last() for y in range(2013, 2026)}
rev = {y: pd.concat([tc.sec_frame(t, f"CY{y}") for t in tc.REVENUE_TAGS]).groupby("cik").val.max() for y in range(2013, 2026)}
num = lambda s: pd.to_numeric(s, errors="coerce").astype(float)
ni = {y: num(s) for y, s in ni.items()}; rev = {y: num(s) for y, s in rev.items()}

def get(d, cik, y):
    return d[y].get(cik, np.nan) if y in d else np.nan
rows = []
fpiv = pf.pivot_table(index="cik", columns="year", values="val", aggfunc="last")
for cik, r in coh.iterrows():
    y = int(r.year)
    n0 = get(ni, cik, y); n0 = get(ni, cik, y + 1) if np.isnan(n0) else n0
    r0 = get(rev, cik, y); r0 = get(rev, cik, y + 1) if np.isnan(r0) else r0
    f0 = fpiv.at[cik, y]
    f3 = fpiv.at[cik, y + 3] if (y + 3) in fpiv.columns else np.nan
    if np.isnan(f3) and (y + 2) in fpiv.columns and (y + 4) in fpiv.columns:
        f3 = fpiv.at[cik, y + 4] if not np.isnan(fpiv.at[cik, y + 4]) else np.nan
    n3 = get(ni, cik, y + 3)
    rows.append({"cik": cik, "name": r.entityName, "cohort": y, "sic": r.sic, "float0": f0, "ni0": n0, "rev0": r0,
                 "float3": f3, "ni3": n3})
c = pd.DataFrame(rows).dropna(subset=["ni0"])
c["loss_on_arrival"] = c.ni0 < 0
c["pre_revenue"] = c.rev0.fillna(0) < 1e6
c["small"] = c.rev0.fillna(0) < 1e8
c["survived3"] = c.float3.notna()
c["mult3"] = c.float3 / c.float0
c["profitable3"] = c.ni3 > 0
tc.save_table(c, "h5b_sec_cohorts", index=False)
print(f"{len(c):,} new filers with net income on arrival; {c.loss_on_arrival.mean():.0%} were losing money")

In [ ]:
by_year = c.groupby("cohort").agg(n=("cik", "size"), pct_loss=("loss_on_arrival", "mean"))
by_year["pct_loss"] *= 100
by_year["ritter_pct"] = t9.set_index("year").pct_eps_negative.reindex(by_year.index)
def outcome(g):
    s = g[g.survived3]
    return pd.Series({"n": len(g), "survived_3y_pct": 100 * g.survived3.mean(),
                      "median_value_multiple_survivors": s.mult3.median(),
                      "share_survivors_up_pct": 100 * (s.mult3 > 1).mean(),
                      "profitable_after_3y_pct": 100 * g.profitable3.mean()})
out = c.groupby([c.small.map({True: "Revenue < $100m", False: "Revenue >= $100m"}),
                 c.loss_on_arrival.map({True: "Losing money", False: "Profitable"})]).apply(outcome)
out_all = c.groupby(c.loss_on_arrival.map({True: "Losing money", False: "Profitable"})).apply(outcome)
tc.save_table(out, "h5b_outcomes_by_size"); tc.save_table(out_all, "h5b_outcomes")
print(by_year.round(1).to_string()); print(); print(out_all.round(2).to_string())
out.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(by_year.index, by_year.pct_loss, "o-", color=tc.C[0], ms=5, label="New SEC filers (this notebook)")
ax.plot(by_year.index, by_year.ritter_pct, "o--", color=tc.C[1], ms=5, label="IPOs (Ritter)")
ax.set_ylim(0, 100); ax.set_ylabel("% losing money on arrival"); ax.legend(loc="lower left")
ax.set_title("Loss-making share of new public companies", loc="left")
ax = axes[1]
labels = ["Survived 3 years", "Survivors worth more\nafter 3 years", "Profitable after\n3 years"]
cols = ["survived_3y_pct", "share_survivors_up_pct", "profitable_after_3y_pct"]
x = np.arange(len(cols))
for i, g in enumerate(["Losing money", "Profitable"]):
    v = out_all.loc[g, cols].values
    ax.bar(x + (i - 0.5) * 0.38, v, width=0.36, color=tc.C[1] if g == "Losing money" else tc.C[0], label=f"{g} on arrival")
    for xi, vi in zip(x, v): ax.text(xi + (i - 0.5) * 0.38, vi + 1.5, f"{vi:.0f}%", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 105); ax.set_ylabel("% of cohort")
ax.legend(loc="upper right"); ax.set_title("Three years later (cohorts 2013–2022)", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h5_fig3_sec_cohorts")

## H5c · Cleantech after the book: the technology won, did investors?

* **Deployment:** world solar electricity generation (TWh), from Our World in Data's energy dataset.
* **Investor returns** (total return):
  * the clean-energy ETFs PBW (since 2005), ICLN and TAN (both since 2008) and QCLN;
  * the solar manufacturers and installers First Solar, Canadian Solar, JinkoSolar, Enphase, SolarEdge and Sunrun.
* **Windows:** from the book (16 Sep 2014), and from the end of 2007, the first cleantech peak in Thiel's chapter. A fund that started later is measured from its first day.
* SunPower, the largest US solar maker in 2014, went bankrupt in 2024 and its shares were cancelled. It is not in Yahoo's data and is noted separately rather than included.

In [ ]:
import requests
owid_cache = tc.RAW / "owid_energy.csv"
if not owid_cache.exists() and not tc.SYNTHETIC:
    r = requests.get("https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv", timeout=180)
    r.raise_for_status(); owid_cache.write_bytes(r.content)
if tc.SYNTHETIC:
    solar = pd.Series(np.geomspace(30, 2800, 16), index=range(2010, 2026))
else:
    e = pd.read_csv(owid_cache, usecols=["country", "year", "solar_electricity", "solar_share_elec"])
    w = e[e.country == "World"].set_index("year")
    solar = w.solar_electricity.dropna()
    print(f"World solar generation: {solar.get(2007, np.nan):.0f} TWh (2007), {solar[2014]:.0f} TWh (2014), "
          f"{solar.iloc[-1]:.0f} TWh ({solar.index[-1]}); share of electricity {w.solar_share_elec.get(2014):.1f}% -> {w.solar_share_elec.dropna().iloc[-1]:.1f}%")

CLEAN = {"PBW": "WilderHill Clean Energy ETF", "ICLN": "iShares Global Clean Energy", "TAN": "Invesco Solar ETF",
         "QCLN": "First Trust NASDAQ Clean Edge", "FSLR": "First Solar", "CSIQ": "Canadian Solar", "JKS": "JinkoSolar",
         "ENPH": "Enphase Energy", "SEDG": "SolarEdge", "RUN": "Sunrun", "SPY": "S&P 500 (SPY)", "QQQ": "Nasdaq-100 (QQQ)"}
px = {t: tc.yahoo_close(t) for t in CLEAN}
END = px["SPY"].index.max()
def mult(s, start):
    s = s.dropna(); s = s.loc[s.index >= start]
    if len(s) < 20: return np.nan, None
    return s.iloc[-1] / s.iloc[0], s.index[0]
rows = []
for t, name in CLEAN.items():
    for lab, start in [("since book (16 Sep 2014)", pd.Timestamp("2014-09-16")), ("since end-2007 peak", pd.Timestamp("2007-12-31"))]:
        m, s0 = mult(px[t], start)
        if s0 is None: continue
        spy_m, _ = mult(px["SPY"], s0)
        rows.append({"ticker": t, "name": name, "window": lab, "start": s0.date(), "multiple": m, "vs_spy": m / spy_m,
                     "cagr_pct": 100 * (m ** (365.25 / (END - s0).days) - 1)})
ct = pd.DataFrame(rows); tc.save_table(ct, "h5c_cleantech_returns", index=False)
ct.pivot_table(index=["ticker", "name"], columns="window", values=["multiple", "vs_spy"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw={"width_ratios": [1, 1.1]})
ax = axes[0]
y0 = 2014
sol = solar.loc[y0:] / solar[y0]
ax.plot(sol.index, sol.values, color=tc.C[2], lw=2.4, marker="o", ms=4, label=f"World solar generation ({sol.iloc[-1]:.0f}x)")
for i, t in enumerate(["TAN", "ICLN", "SPY"]):
    a = px[t].resample("YE").last(); a.index = a.index.year
    a = a.loc[y0:] / px[t].loc[:"2014-12-31"].iloc[-1]
    col = tc.INK if t == "SPY" else tc.C[[0, 1][i]]
    ax.plot(a.index, a.values, color=col, lw=1.8, ls="--" if t == "SPY" else "-", label=f"{CLEAN[t]} ({a.iloc[-1]:.1f}x)")
ax.set_yscale("log"); ax.axhline(1, color=tc.GREY, lw=0.8)
from matplotlib.ticker import FuncFormatter, NullFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}x")); ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylabel("Index, end of 2014 = 1 (log)"); ax.legend(loc="upper left", fontsize=8)
ax.set_title("Solar output vs solar investors since 2014", loc="left")
ax = axes[1]
d = ct[(ct.window == "since book (16 Sep 2014)") & ~ct.ticker.isin(["SPY", "QQQ"])].sort_values("vs_spy")
ax.barh(range(len(d)), d.vs_spy, color=[tc.C[0] if t in ("PBW", "ICLN", "TAN", "QCLN") else tc.C[1] for t in d.ticker], height=0.7)
ax.axvline(1, color=tc.INK, ls=":", lw=1); ax.set_xscale("log")
ax.set_yticks(range(len(d))); ax.set_yticklabels([f"{n} ({t})" for t, n in zip(d.ticker, d.name)], fontsize=8)
xmax = max(4, d.vs_spy.max() * 1.5); ax.set_xlim(min(0.02, d.vs_spy.min() * 0.7), xmax * 2)
for i, m in enumerate(d.multiple): ax.text(xmax * 1.1, i, f"{m:.2f}x", va="center", fontsize=8)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}x"))
ax.set_xlabel("Multiple relative to the S&P 500 since 16 Sep 2014 (log); right: $1 became")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=tc.C[0], label="ETF"), Patch(color=tc.C[1], label="Company")], loc="upper center",
          bbox_to_anchor=(0.45, -0.16), ncol=2, fontsize=8)
ax.set_title("Clean-energy stocks against the index", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h5_fig4_cleantech")

## Verdict inputs and check cell — send me this output

In [ ]:
def rec(df):
    return json.loads(df.round(3).to_json(orient="records", date_format="iso"))
summary = {
    "eras": rec(era),
    "ritter_peak_years": t9[t9.pct_eps_negative >= 75][["year", "pct_eps_negative"]].values.tolist(),
    "ritter_t16b": rec(t16),
    "sec_cohorts_n": int(len(c)), "sec_pct_loss": round(float(c.loss_on_arrival.mean() * 100), 1),
    "sec_by_year": json.loads(by_year.round(1).to_json(orient="index")),
    "sec_outcomes": json.loads(out_all.round(2).to_json(orient="index")),
    "sec_outcomes_by_size": {f"{a}|{b}": v for (a, b), v in out.round(2).to_dict("index").items()},
    "solar_twh": {int(k): round(float(v), 1) for k, v in solar.loc[2007:].items()},
    "cleantech": rec(ct),
}
(tc.OUT / "h5_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str)[:6000])